In [1]:
import sys
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime, timedelta

project_root = Path.cwd().parents[1]
sys.path.insert(0, str(project_root / 'common' / 'scripts'))
from metabase_client import MetabaseClient

# ── Parameters ──
NAV_DATE = '2026-03-02'
FUND_CODE = 'TUK75'
FUND_NAME = 'Tuleva Maailma Aktsiate Pensionifond'
FUND_ISIN = 'EE3600109435'

client = MetabaseClient()
print(f'NAV date: {NAV_DATE}  |  Fund: {FUND_CODE} ({FUND_NAME})')

NAV date: 2026-03-02  |  Fund: TUK75 (Tuleva Maailma Aktsiate Pensionifond)


In [2]:
# ── Fetch all data from Metabase ──
raw_positions = pd.DataFrame(client.execute_card(2296))
raw_index = pd.DataFrame(client.execute_card(2297))
raw_units = pd.DataFrame(client.execute_card(2298))
raw_registry = pd.DataFrame(client.execute_card(2299))

print(f'Card 2296 (positions):  {len(raw_positions)} rows')
print(f'Card 2297 (index):      {len(raw_index)} rows')
print(f'Card 2298 (units):      {len(raw_units)} rows')
print(f'Card 2299 (registry):   {len(raw_registry)} rows')

# ── Filter positions to our fund and date ──
all_positions = raw_positions[
    (raw_positions['Fund Code'] == FUND_CODE) &
    (raw_positions['Nav Date'] == NAV_DATE)
].copy()

# Categorize by Account Type
securities = all_positions[all_positions['Account Type'] == 'SECURITY'].copy()
cash = all_positions[all_positions['Account Type'] == 'CASH'].copy()
receivables = all_positions[all_positions['Account Type'] == 'RECEIVABLES'].copy()
liabilities = all_positions[all_positions['Account Type'] == 'LIABILITY'].copy()
units_row = all_positions[all_positions['Account Type'] == 'UNITS'].copy()

print(f'\nPositions for {FUND_CODE} on {NAV_DATE}: {len(all_positions)} rows')
print(f'  SECURITY:     {len(securities)}')
print(f'  CASH:         {len(cash)}')
print(f'  RECEIVABLES:  {len(receivables)}')
print(f'  LIABILITY:    {len(liabilities)}')
print(f'  UNITS:        {len(units_row)}')

# ── Filter units: use all available dates for this fund ──
all_fund_units = raw_units[raw_units['Security Name'] == FUND_NAME].copy()
all_fund_units = all_fund_units.sort_values('Request Date', ascending=False)

units_today = all_fund_units[all_fund_units['Request Date'] == NAV_DATE]

# Find the last date where NAV actually changed (skip weekends with same NAV)
prev_dates = all_fund_units[all_fund_units['Request Date'] < NAV_DATE]['Request Date'].unique()
prev_date = None
if len(prev_dates) > 0 and len(units_today) > 0:
    today_nav_val = units_today.iloc[0]['Nav']
    for d in sorted(prev_dates, reverse=True):
        row = all_fund_units[all_fund_units['Request Date'] == d].iloc[0]
        if row['Nav'] != today_nav_val:
            prev_date = d
            break
    if prev_date is None:
        prev_date = sorted(prev_dates)[-1]

units_prev = all_fund_units[all_fund_units['Request Date'] == prev_date] if prev_date else pd.DataFrame()
print(f'\nUnits dates: today={NAV_DATE}, prev={prev_date}')

# ── Look up management fee rate from registry ──
fund_reg = raw_registry[raw_registry['Isin'] == FUND_ISIN]
if len(fund_reg) > 0:
    mgmt_fee_rate = fund_reg.iloc[0]['Management Fee Rate']
    print(f'Management fee rate: {mgmt_fee_rate} ({mgmt_fee_rate*100:.3f}% p.a.)')
else:
    mgmt_fee_rate = None
    print('WARNING: Fund not found in registry!')

Card 2296 (positions):  12 rows
Card 2297 (index):      511 rows
Card 2298 (units):      18 rows
Card 2299 (registry):   58 rows

Positions for TUK75 on 2026-03-02: 12 rows
  SECURITY:     6
  CASH:         1
  RECEIVABLES:  2
  LIABILITY:    2
  UNITS:        1

Units dates: today=2026-03-02, prev=2026-02-28
Management fee rate: 0.00205 (0.205% p.a.)


## Fund Position

All holdings of the fund on the NAV date, split into securities (ETFs) and cash/accrual lines.

In [3]:
# ── Securities (ETF holdings) ──
sec_display = securities[['Account ID', 'Account Name', 'Quantity', 'Market Price', 'Market Value']].copy()
sec_display = sec_display.sort_values('Market Value', ascending=False)
sec_total = sec_display['Market Value'].sum()

display(sec_display.style
    .format({'Quantity': '{:,.2f}', 'Market Price': '{:.4f}', 'Market Value': '{:,.2f}'})
    .hide(axis='index'))
print(f'Securities total: {sec_total:,.2f} EUR')

# ── Cash ──
cash_total = cash['Market Value'].sum()
print(f'\nCash: {cash_total:,.2f} EUR')
for _, r in cash.iterrows():
    print(f'  {r["Account Name"]}: {r["Market Value"]:,.2f} EUR')

# ── Receivables & Liabilities ──
recv_total = receivables['Market Value'].sum()
liab_total = liabilities['Market Value'].sum()

print(f'\nReceivables: {recv_total:,.2f} EUR')
for _, r in receivables.iterrows():
    print(f'  {r["Account Name"]}: {r["Market Value"]:,.2f} EUR')

print(f'\nLiabilities: {liab_total:,.2f} EUR')
for _, r in liabilities.iterrows():
    print(f'  {r["Account Name"]}: {r["Market Value"]:,.2f} EUR')

# ── Total units from position data ──
if len(units_row) > 0:
    position_units = units_row.iloc[0]['Quantity']
    print(f'\nTotal outstanding units (from position): {position_units:,.3f}')

# ── Net Asset Value ──
nav_components = sec_total + cash_total + recv_total + liab_total
print(f'\n── NAV Components ──')
print(f'  Securities:    {sec_total:>16,.2f} EUR')
print(f'  Cash:          {cash_total:>16,.2f} EUR')
print(f'  Receivables:   {recv_total:>16,.2f} EUR')
print(f'  Liabilities:   {liab_total:>16,.2f} EUR')
print(f'  Total:         {nav_components:>16,.2f} EUR')

Account ID,Account Name,Quantity,Market Price,Market Value
IE0009FT4LX4,CCF Developed World (ESG Screened) Index Fund Class X0,"17,953,577.58",15.4510,"277,400,727.13"
IE00BFG1TM61,iShares Developed World Screened Index Fund,"8,042,414.77",34.3430,"276,200,650.45"
IE00BFNM3G45,iShares MSCI USA Screened UCITS ETF,"17,609,096.00",12.0480,"212,154,388.61"
IE00BKPTWY98,iShares Emerging Market Screened Equity Index Fund Class Flexible,"7,550,798.54",13.7700,"103,974,495.90"
IE00BFNM3D14,iShares MSCI Europe Screened UCITS ETF,"7,026,993.00",10.4180,"73,207,213.07"
IE00BFNM3L97,iShares MSCI Japan Screened UCITS ETF,"1,034,931.00",7.8280,"8,101,439.87"


Securities total: 951,038,915.03 EUR

Cash: 1,633,975.32 EUR
  Cash account in SEB Pank: 1,633,975.32 EUR

Receivables: 0.00 EUR
  Total receivables of unsettled transactions: 0.00 EUR
  Receivables of outstanding units: 0.00 EUR

Liabilities: 0.00 EUR
  Total payables of unsettled transactions: 0.00 EUR
  Payables of redeemed units: 0.00 EUR

Total outstanding units (from position): 667,238,758.286

── NAV Components ──
  Securities:      951,038,915.03 EUR
  Cash:              1,633,975.32 EUR
  Receivables:               0.00 EUR
  Liabilities:               0.00 EUR
  Total:           952,672,890.35 EUR


## Price Verification

Cross-checking fund position prices against independent index/provider data. Prices are flagged if the difference exceeds 0.5%.

In [4]:
# ── Parse index data: extract ISIN from Key column (format: ISIN.PROVIDER) ──
idx = raw_index.copy()
idx['Date'] = pd.to_datetime(idx['Date'])
nav_dt = pd.to_datetime(NAV_DATE)

# Split Key into ISIN and Provider parts
idx[['Key_ISIN', 'Key_Provider']] = idx['Key'].str.split('.', n=1, expand=True)

# For each ETF holding, find the latest index value on or before NAV_DATE
verify_rows = []
for _, sec in securities.iterrows():
    isin = sec['Account ID']
    fund_price = sec['Market Price']

    # Find matching index entries for this ISIN
    matches = idx[(idx['Key_ISIN'] == isin) & (idx['Date'] <= nav_dt)]
    if len(matches) == 0:
        verify_rows.append({
            'ISIN': isin,
            'Fund Price': fund_price,
            'Index Price': None,
            'Provider': None,
            'Index Date': None,
            'Diff %': None,
            'Status': 'NO DATA',
        })
        continue

    # Take the latest date
    latest = matches.loc[matches['Date'].idxmax()]
    idx_price = latest['Value']
    diff_pct = (fund_price - idx_price) / idx_price * 100 if idx_price else None

    verify_rows.append({
        'ISIN': isin,
        'Fund Price': fund_price,
        'Index Price': idx_price,
        'Provider': latest.get('Key_Provider') or latest.get('Provider'),
        'Index Date': latest['Date'].strftime('%Y-%m-%d'),
        'Diff %': diff_pct,
        'Status': 'OK' if diff_pct is not None and abs(diff_pct) <= 0.5 else 'FLAG',
    })

verify_df = pd.DataFrame(verify_rows)

def color_status(val):
    if val == 'OK':
        return 'background-color: #d4edda'
    elif val == 'FLAG':
        return 'background-color: #f8d7da'
    return 'background-color: #fff3cd'

display(verify_df.style
    .format({'Fund Price': '{:.4f}', 'Index Price': '{:.4f}', 'Diff %': '{:+.3f}%'},
            na_rep='—')
    .map(color_status, subset=['Status'])
    .hide(axis='index'))

n_ok = (verify_df['Status'] == 'OK').sum()
n_flag = (verify_df['Status'] == 'FLAG').sum()
n_nodata = (verify_df['Status'] == 'NO DATA').sum()
print(f'{n_ok} OK  |  {n_flag} flagged  |  {n_nodata} no data')

ISIN,Fund Price,Index Price,Provider,Index Date,Diff %,Status
IE00BFG1TM61,34.3430,34.3430,MORNINGSTAR,2026-02-27,+0.000%,OK
IE00BKPTWY98,13.7700,13.7700,MORNINGSTAR,2026-02-27,+0.000%,OK
IE0009FT4LX4,15.4510,15.4510,MORNINGSTAR,2026-02-27,+0.000%,OK
IE00BFNM3G45,12.0480,11.9440,XETR,2026-02-27,+0.871%,FLAG
IE00BFNM3L97,7.8280,7.9850,XETR,2026-02-27,-1.966%,FLAG
IE00BFNM3D14,10.4180,10.6020,XETR,2026-02-27,-1.736%,FLAG


3 OK  |  3 flagged  |  0 no data


## Net Asset Value Calculation

Computing NAV per unit from position data and comparing to the reported value.

In [5]:
nav_dt = datetime.strptime(NAV_DATE, '%Y-%m-%d')

# ── Total units: from UNITS row in position data ──
position_units = units_row.iloc[0]['Quantity'] if len(units_row) > 0 else None

# ── From units card ──
row_today = units_today.iloc[0]
reported_nav = row_today['Nav']
reported_balance = row_today['Balance']
count_units = row_today['Count Units']
count_units_fm = row_today['Count Units Fm']
card_total_units = count_units + count_units_fm

print('── Units Reconciliation ──')
print(f'  Position UNITS row:    {position_units:>16,.3f}')
print(f'  Card: Count Units:     {count_units:>16,.3f}')
print(f'  Card: Count Units Fm:  {count_units_fm:>16,.3f}')
print(f'  Card: Total:           {card_total_units:>16,.3f}')
print(f'  Difference:            {position_units - card_total_units:>16,.3f}')

# ── NAV Calculation ──
# NAV = (Securities + Cash + Receivables + Liabilities) / Total Units
computed_nav = nav_components / position_units

nav_diff_eur = computed_nav - reported_nav
nav_diff_pct = nav_diff_eur / reported_nav * 100

print(f'\n── NAV Calculation ──')
print(f'  Net assets:            {nav_components:>16,.2f} EUR')
print(f'  Total units:           {position_units:>16,.3f}')
print(f'  Computed NAV/unit:     {computed_nav:>16.6f} EUR')
print(f'  Reported NAV/unit:     {reported_nav:>16.6f} EUR')
print(f'  Difference:            {nav_diff_eur:>16.6f} EUR ({nav_diff_pct:+.4f}%)')

# ── Cross-check: Balance vs computed ──
expected_balance = reported_nav * card_total_units
print(f'\n── Balance Cross-check ──')
print(f'  Reported Balance:      {reported_balance:>16,.2f} EUR')
print(f'  NAV × Total Units:     {expected_balance:>16,.2f} EUR')
print(f'  Difference:            {reported_balance - expected_balance:>16,.2f} EUR')
print(f'  Net assets − Balance:  {nav_components - reported_balance:>16,.2f} EUR')

# ── Fee accrual estimate ──
day_of_month = nav_dt.day
expected_daily_fee = nav_components * mgmt_fee_rate / 365
expected_accrual = expected_daily_fee * day_of_month

print(f'\n── Fee Accrual Estimate ──')
print(f'  Mgmt fee rate:     {mgmt_fee_rate*100:.3f}% p.a.')
print(f'  Daily fee:         {expected_daily_fee:>12,.2f} EUR')
print(f'  Accrual ({day_of_month} days):  {expected_accrual:>12,.2f} EUR')

# ── Verdict ──
net_assets = nav_components
total_units = position_units
status = 'PASS' if abs(nav_diff_eur) < 0.0001 else 'MISMATCH'
print(f'\n  Verification: {status} (diff: {nav_diff_eur:+.6f} EUR / {nav_diff_pct:+.4f}%)')

── Units Reconciliation ──
  Position UNITS row:     667,238,758.286
  Card: Count Units:      661,491,407.286
  Card: Count Units Fm:     5,747,351.000
  Card: Total:            667,238,758.286
  Difference:                       0.000

── NAV Calculation ──
  Net assets:              952,672,890.35 EUR
  Total units:            667,238,758.286
  Computed NAV/unit:             1.427784 EUR
  Reported NAV/unit:             1.430970 EUR
  Difference:                   -0.003186 EUR (-0.2226%)

── Balance Cross-check ──
  Reported Balance:        954,798,645.94 EUR
  NAV × Total Units:       954,798,645.94 EUR
  Difference:                       -0.00 EUR
  Net assets − Balance:     -2,125,755.59 EUR

── Fee Accrual Estimate ──
  Mgmt fee rate:     0.205% p.a.
  Daily fee:             5,350.63 EUR
  Accrual (2 days):     10,701.26 EUR

  Verification: MISMATCH (diff: -0.003186 EUR / -0.2226%)


## Day-over-Day Comparison

Comparing the fund's daily NAV change to the benchmark index movement.

In [6]:
# ── Previous day NAV ──
if len(units_prev) == 0 or len(units_today) == 0:
    print(f'Cannot compare: missing data for {prev_date} or {NAV_DATE}')
else:
    prev_nav = units_prev.iloc[0]['Nav']
    today_nav = reported_nav
    nav_change_pct = (today_nav - prev_nav) / prev_nav * 100

    print(f'── Fund NAV Change ──')
    print(f'  {prev_date}: {prev_nav:.6f}')
    print(f'  {NAV_DATE}:  {today_nav:.6f}')
    print(f'  Change:      {nav_change_pct:+.4f}%')

    # ── Index change ──
    idx_data = raw_index.copy()
    idx_data['Date'] = pd.to_datetime(idx_data['Date'])

    # Try common index key patterns
    index_keys = [k for k in idx_data['Key'].unique()
                  if any(x in k.upper() for x in ['MSCI_ACWI', 'GLOBAL_STOCK', 'ACWI'])]

    if index_keys:
        idx_key = index_keys[0]
        idx_series = idx_data[idx_data['Key'] == idx_key].sort_values('Date')

        nav_dt = pd.to_datetime(NAV_DATE)
        prev_dt = pd.to_datetime(prev_date)

        idx_on_nav = idx_series[idx_series['Date'] <= nav_dt]
        idx_on_prev = idx_series[idx_series['Date'] <= prev_dt]

        if len(idx_on_nav) > 0 and len(idx_on_prev) > 0:
            idx_today_row = idx_on_nav.iloc[-1]
            idx_prev_row = idx_on_prev.iloc[-1]

            # Only compare if the index dates are actually different
            if idx_today_row['Date'] != idx_prev_row['Date']:
                idx_change_pct = (idx_today_row['Value'] - idx_prev_row['Value']) / idx_prev_row['Value'] * 100
                tracking_diff = nav_change_pct - idx_change_pct
            else:
                idx_change_pct = 0.0
                tracking_diff = nav_change_pct

            print(f'\n── Benchmark Index ({idx_key}) ──')
            print(f'  {idx_prev_row["Date"].strftime("%Y-%m-%d")}: {idx_prev_row["Value"]:.4f}')
            print(f'  {idx_today_row["Date"].strftime("%Y-%m-%d")}: {idx_today_row["Value"]:.4f}')
            print(f'  Change:      {idx_change_pct:+.4f}%')

            print(f'\n── Tracking Difference ──')
            comparison = pd.DataFrame([{
                'Metric': 'Fund NAV change',
                'Value': f'{nav_change_pct:+.4f}%',
            }, {
                'Metric': f'Index change ({idx_key})',
                'Value': f'{idx_change_pct:+.4f}%',
            }, {
                'Metric': 'Tracking difference',
                'Value': f'{tracking_diff:+.4f}%',
            }])
            display(comparison.style.hide(axis='index'))
        else:
            print(f'\nIndex data insufficient for comparison')
    else:
        non_isin_keys = [k for k in idx_data['Key'].unique() if '.' not in k]
        print(f'\nNo MSCI ACWI index found. Available non-ISIN keys: {non_isin_keys}')

── Fund NAV Change ──
  2026-02-28: 1.434800
  2026-03-02:  1.430970
  Change:      -0.2669%

── Benchmark Index (MSCI_ACWI) ──
  2026-02-27: 462.6001
  2026-02-27: 462.6001
  Change:      +0.0000%

── Tracking Difference ──


Metric,Value
Fund NAV change,-0.2669%
Index change (MSCI_ACWI),+0.0000%
Tracking difference,-0.2669%


## Summary

In [7]:
# ── Summary table ──
summary_rows = []

summary_rows.append({'Item': 'NAV Date', 'Value': NAV_DATE})
summary_rows.append({'Item': 'Fund', 'Value': f'{FUND_CODE} — {FUND_NAME}'})
summary_rows.append({'Item': '', 'Value': ''})
summary_rows.append({'Item': 'Securities', 'Value': f'{sec_total:,.2f} EUR'})
summary_rows.append({'Item': 'Cash', 'Value': f'{cash_total:,.2f} EUR'})
summary_rows.append({'Item': 'Receivables', 'Value': f'{recv_total:,.2f} EUR'})
summary_rows.append({'Item': 'Liabilities', 'Value': f'{liab_total:,.2f} EUR'})
summary_rows.append({'Item': 'Net assets', 'Value': f'{nav_components:,.2f} EUR'})
summary_rows.append({'Item': 'Reported Balance', 'Value': f'{reported_balance:,.2f} EUR'})
summary_rows.append({'Item': 'Net assets − Balance', 'Value': f'{nav_components - reported_balance:,.2f} EUR'})
summary_rows.append({'Item': '', 'Value': ''})
summary_rows.append({'Item': 'Total units', 'Value': f'{total_units:,.3f}'})
summary_rows.append({'Item': 'Computed NAV/unit', 'Value': f'{computed_nav:.6f} EUR'})
summary_rows.append({'Item': 'Reported NAV/unit', 'Value': f'{reported_nav:.6f} EUR'})
summary_rows.append({'Item': 'NAV difference', 'Value': f'{nav_diff_eur:+.6f} EUR ({nav_diff_pct:+.4f}%)'})

# Daily comparison
if len(units_prev) > 0:
    summary_rows.append({'Item': '', 'Value': ''})
    summary_rows.append({'Item': f'Previous NAV date', 'Value': prev_date})
    summary_rows.append({'Item': 'Daily NAV change', 'Value': f'{nav_change_pct:+.4f}%'})
    if index_keys:
        summary_rows.append({'Item': f'Index change ({idx_key})', 'Value': f'{idx_change_pct:+.4f}%'})
        summary_rows.append({'Item': 'Tracking difference', 'Value': f'{tracking_diff:+.4f}%'})

summary_rows.append({'Item': '', 'Value': ''})
summary_rows.append({'Item': 'Price checks passed', 'Value': f'{n_ok} / {len(verify_df)}'})
summary_rows.append({'Item': 'Est. fee accrual', 'Value': f'{expected_accrual:,.2f} EUR'})

nav_ok = abs(nav_diff_eur) < 0.0001
summary_rows.append({'Item': 'NAV VERIFICATION', 'Value': 'PASS' if nav_ok else 'MISMATCH'})

summary_df = pd.DataFrame(summary_rows)

def highlight_verdict(row):
    if row['Item'] == 'NAV VERIFICATION':
        color = '#d4edda' if row['Value'] == 'PASS' else '#f8d7da'
        return [f'background-color: {color}; font-weight: bold'] * 2
    return [''] * 2

display(summary_df.style
    .apply(highlight_verdict, axis=1)
    .hide(axis='index'))

Item,Value
NAV Date,2026-03-02
Fund,TUK75 — Tuleva Maailma Aktsiate Pensionifond
,
Securities,"951,038,915.03 EUR"
Cash,"1,633,975.32 EUR"
Receivables,0.00 EUR
Liabilities,0.00 EUR
Net assets,"952,672,890.35 EUR"
Reported Balance,"954,798,645.94 EUR"
Net assets − Balance,"-2,125,755.59 EUR"
